# Pretraining & Alignment

- [Pretraining vs. Fine-Tuning](#Pretraining-vs.-Fine-Tuning)
  - [Base vs. instruct/chat: worked comparison](#Base-vs.-instruct/chat:-worked-comparison)
  - [Llama-2 base vs. chat, and ChatGPT](#Llama-2-base-vs.-chat,-and-ChatGPT)
  - [Non-finetuned models](#Non-finetuned-models)
  - [Finetuned models](#Finetuned-models)
  - [Comparison with ChatGPT](#Comparison-with-ChatGPT)
  - [Scaling laws](#Scaling-laws)
- [Supervised Fine-Tuning (SFT) and PEFT](#Supervised-Fine-Tuning-(SFT)-and-PEFT)
  - [SFT: imitating example responses](#SFT:-imitating-example-responses)
  - [SFT data curation](#SFT-data-curation)
  - [Full fine-tuning vs. PEFT](#Full-fine-tuning-vs.-PEFT)
  - [SFT on a small model](#SFT-on-a-small-model)
  - [Testing the trained small model](#Testing-the-trained-small-model)
  - [PEFT beyond LoRA: QLoRA and prompt tuning](#PEFT-beyond-LoRA:-QLoRA-and-prompt-tuning)
  - [Catastrophic forgetting](#Catastrophic-forgetting)
  - [Fine-tuning vs feature extraction](#Fine-tuning-vs-feature-extraction)
- [LoRA (Low-Rank Adaptation)](#LoRA-(Low-Rank-Adaptation))
  - [Hands-on LoRA](#Hands-on-LoRA)
  - [Serving and merging LoRA adapters](#Serving-and-merging-LoRA-adapters)
  - [Model merging and distillation](#Model-merging-and-distillation)
- [Alignment and RLHF](#Alignment-and-RLHF)
  - [Alignment gaps](#Alignment-gaps)
  - [The three-act story](#The-three-act-story)
  - [Act 1: SFT](#Act-1:-SFT)
  - [Act 2: a proxy for human judgement](#Act-2:-a-proxy-for-human-judgement)
  - [Bradley-Terry, implemented](#Bradley-Terry,-implemented)
  - [Act 3: using RL to maximise the proxy](#Act-3:-using-RL-to-maximise-the-proxy)
  - [A brief history: from robots to language models](#A-brief-history:-from-robots-to-language-models)
  - [The villain: reward hacking](#The-villain:-reward-hacking)
  - [DPO: skipping the reward model](#DPO:-skipping-the-reward-model)
  - [Current research](#Current-research)
  - [RLAIF](#RLAIF)

### Pretraining vs. Fine-Tuning

How a base (pretrained-only) model becomes an instruction-following/chat model: base vs. instruct comparisons, supervised fine-tuning (SFT) mechanics, full fine-tuning vs. parameter-efficient fine-tuning (PEFT), and LoRA as the specific PEFT technique used below.

#### Base vs. instruct/chat: worked comparison

Same base model, run with and without instruction fine-tuning, on identical prompts. A base model (pretrained only, next-token prediction on raw text) tends to continue a prompt rather than answer it; a fine-tuned/instruct model has been further trained on instruction-response pairs so it follows the instruction instead. Two worked examples below: Qwen2.5-0.5B (base vs. instruct, self-contained, runs locally) and Llama-2 (base vs. chat, via the Lamini API + a `llama.py` runner, needs external credentials to actually execute, kept for the comparison it documents even though it won't run standalone here).

In [1]:
### Pretraining vs. Fine-tuning

Base (pretrained-only) vs. instruct (fine-tuned) Qwen2.5-0.5B run side by side on identical prompts.
! pip install torch==2.9.0 transformers==4.57.3 matplotlib==3.10.0 pandas==2.2.2 numpy==2.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface_hub-1.4.0:
      Successfully uninstalled huggingface_hub-1.4.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_ID = "Qwen/Qwen2.5-0.5B"
INST_ID = "Qwen/Qwen2.5-0.5B-Instruct"

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype="auto",
        device_map="auto",
        trust_remote_code=True
    )
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return tok, mdl

base_tok, base_m = load(BASE_ID)
inst_tok, inst_m = load(INST_ID)

@torch.inference_mode()
def gen_base(prompt, max_new_tokens=80):
    x = base_tok(prompt, return_tensors="pt").to(base_m.device)
    y = base_m.generate(
        **x,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=base_tok.eos_token_id,
    )
    return base_tok.decode(y[0], skip_special_tokens=True)

@torch.inference_mode()
def gen_instruct(user_prompt, max_new_tokens=80):
    # Proper chat formatting for instruct model
    chat = [{"role": "user", "content": user_prompt}]
    prompt = inst_tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    x = inst_tok(prompt, return_tensors="pt").to(inst_m.device)
    y = inst_m.generate(
        **x,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=inst_tok.eos_token_id,
    )
    out = inst_tok.decode(y[0], skip_special_tokens=True)
    # Trim the echoed prompt if present
    return out[len(inst_tok.decode(x["input_ids"][0], skip_special_tokens=True)) :].strip()

tests = [
    "How do I make a pizza?",
    "Write a haiku about summer.",
]

for p in tests:
    print("\n" + "="*90)
    print("PROMPT:", p)
    print("-"*90)
    print("BASE (pretrained):")
    print(gen_base(p))
    print("-"*90)
    print("INSTRUCT (fine-tuned):")
    print(gen_instruct(p))


#### Llama-2 base vs. chat, and ChatGPT

Same comparison, different model family and toolchain (Lamini API + local `llama.py` runner, not
self-contained like the Qwen example above — needs `POWERML__PRODUCTION__URL`/`KEY` env vars and a
`../utils/llama.py` module to actually execute).

In [12]:
import os
import lamini

lamini.api_url = os.getenv("POWERML__PRODUCTION__URL")
lamini.api_key = os.getenv("POWERML__PRODUCTION__KEY")

In [14]:
import sys
import os

sys.path.append('../utils')
#print(os.path.exists('../utils/llama.py'))  # Should print True
#print(os.listdir('../utils'))  # Shows files in the utils folder

from llama import BasicModelRunner

#### Non-finetuned models

In [15]:
non_finetuned = BasicModelRunner("meta-llama/Llama-2-7b-hf")

In [16]:
non_finetuned_output = non_finetuned("Tell me how to train my dog to sit")

In [17]:
print(non_finetuned_output)

Tell me how to train my dog to stay.
Tell me how to teach my dog to come.
Tell me how to get my dog to heel.
Tell me how to stop my dog from jumping.
Tell me how to make my dog stop barking.
Tell me how I can get my dog to stop chewing.
Tell me how to house train my dog.
Tell me how to potty train my dog.
Tell me how to train my dog to walk on a leash.
Tell me how to crate train my dog.
Tell Me How To Train My Dog To Sit, Stay, Come, Heel, Stop Jumping, Stop Barking, Stop Chewing, House Train, Potty Train, Walk On A Leash, Crate Train, And More!
Tell Me How To Train Your Dog To Sit, Stay, Heel, Come, Stop Jumping, Stop Chewing, House Trained, Potty Trained, Walk On A Leash, And More!
Tell me how to train your dog to sit.
Tell Me How To House Train My Dog.


In [18]:
print(non_finetuned("What do you think of Mars?"))

I think it's a great planet.
I think it's a good planet.
I think it'll be a great planet.
I think we should go there.
I think we should go back there.
I think we should stay there.
I think we should leave there.
I think we should colonize there.
I think we should terraform there.
I think we should mine there.
I think we should build there.
I think we should live there.
I think we should die there.
I think we should be there.
I think we should have been there.
I think we should never be there.
I think we'll be there.
I think we won't be there.
I think we will be there.
I think we can be there.
I think we could be there.
I think we would be there.
I think we might be there.
I think we may be there.
I think we shall be there.
I think we must be there.
I think we have to be there.
I think we need to be there.
I think I'll be there.
I know I'll be there.


In [19]:
print(non_finetuned("taylor swift's best friend"))

I'm not sure if I've mentioned this before, but I'm a huge Taylor Swift fan.
I've been a fan since her first album, and I've been a fan ever since. I've been a fan of her music, her style, her personality, and her music. I've been a fan for a long time.
I've been a fan of Taylor Swift for a long time. I've been a fan of her music for a long time. I'm a fan of her style, her personality, her music, and her music. I've been a fan for a long, long time.
I've always been a fan of Taylor Swift. I've always been a fan of her music.
I've always been a big fan of her style. I've always been a huge fan of her personality. I've always been a massive fan of her music. I'm a fan of Taylor Swift. I love her music. I love her style. I love her personality. I love her music. I'm a huge fan of Taylor Swift.
I've always been an avid fan of Taylor Swift. I'm


In [20]:
print(non_finetuned("""Agent: I'm here to help you with your Amazon deliver order.
Customer: I didn't get my item
Agent: I'm sorry to hear that. Which item was it?
Customer: the blanket
Agent:"""))

I'm sorry to hear that but I'm here to help.
Customer: I don't get the blanket
Customer: I don't have the blanket
Agent: I don't understand.
Agent: I'm sorry. I don't understand. customer: I don't have the blanket Agent: I'm so sorry to hear that.
Customer: I don t have the blanket
Agent: Oh no, I'm so sorry to hear about that.
customer: I


#### Finetuned models

In [21]:
finetuned_model = BasicModelRunner("meta-llama/Llama-2-7b-chat-hf")

In [22]:
finetuned_output = finetuned_model("Tell me how to train my dog to sit")

In [23]:
print(finetuned_output)

on command. Training your dog to sit is a basic obedience command that can be achieved with patience, consistency, and positive reinforcement. Here's a step-by-step guide on how to train your dog to sit:

1. Choose a quiet and distraction-free area: Find a quiet room or area where your dog can focus on you without getting distracted.
2. Have treats ready: Choose your dog's favorite treats and have them readily available to reward good behavior.
3. Stand in front of your dog: Stand in front of your dog and hold a treat close to their nose.
4. Move the treat up and back: Slowly move the treat upwards and backwards, towards your dog's tail, while saying "sit" in a calm and clear voice.
5. Dog will follow the treat: As you move the treat, your dog should naturally sit down to follow it. The moment they touch their bottom to the ground, give them the treat and praise them.


In [24]:
print(finetuned_model("[INST]Tell me how to train my dog to sit[/INST]"))

Training your dog to sit is a basic obedience command that can be achieved with patience, consistency, and positive reinforcement. Here's a step-by-step guide on how to train your dog to sit:

1. Choose a quiet and distraction-free area: Find a quiet room or area where your dog can focus on you without getting distracted.
2. Have treats ready: Choose your dog's favorite treats and have them readily available to reward good behavior.
3. Stand in front of your dog: Stand in front of your dog and hold a treat close to their nose.
4. Move the treat up and back: Slowly move the treat upwards and backwards, towards your dog's tail, while saying "sit" in a calm and clear voice.
5. Dog will follow the treat: As you move the treat, your dog should naturally sit down to follow it. The moment they touch their bottom to the ground, give them the treat and praise them.
6. Repeat the process: Repeat steps 3-5 several times, so your dog learns to associate the command "sit" with the action of sitting

In [25]:
print(non_finetuned("[INST]Tell me how to train my dog to sit[/INST]"))

[INST]Tell me how to train my dog to sit[/INST]


In [26]:
print(finetuned_model("What do you think of Mars?"))

Mars is a fascinating planet that has captured the imagination of humans for centuries. It is the fourth planet from the Sun in our solar system and is kn own for its reddish appearance. Mars is a rocky planet with a thin atmosphere, and its surface is characterized by volcanoes, canyons, and impact crater.
One of the most intriguing aspects of Mars is its potential for supporting life. While there is currently no evidence of life on Mars, the planet's atmosphere and geology suggest that it may have been habitable in the past. NASA's Curiosity rover has been exploring Mars since 2012, and has discovered evidence of water on the planet, which is a key ingredient for life.
Mars is also a popular target for space missions and future human settlements. Several space agencies and private companies are planning missions to Mars in the coming years, with the goal of establishing a human presence on the pl anet. The challenges of establishing a human settlement on Mars are significa nt, includ

In [27]:
print(finetuned_model("taylor swift's best friend"))

Taylor Swift's best friend is a person who has been by her side through thick and thin. Here are some possible candidates:
1. Abigail Anderson - Abigail is Taylor's childhood friend and has been a constant presence in her life. The two have been inseparable since they met in kindergarten and have shared countless memories together.
2. Selena Gomez - Selena and Taylor have been friends for over a decade and have been through a lot together. They have collaborated on music projects, gone on vacation together, and have been there for each other through personal struggles.
3. Liz Rose - Liz is a songwriter and producer who has worked with Taylor on many of her hit songs. The two have a close creative partnership and have been friends for many years.
4. Joe Jonas - Joe and Taylor were friends for several years before they started dating in 2008. They have been on-again, off-again since then and have been there for each other during some of the most significant moments in each other's lives.

In [28]:
print(finetuned_model("""Agent: I'm here to help you with your Amazon deliver order.
Customer: I didn't get my item
Agent: I'm sorry to hear that. Which item was it?
Customer: the blanket
Agent:"""))

I see. Can you please provide me with your order number so I can look into this for you?
Customer: I don't have the order number.
Agent: Okay, no worries. Can you please tell me the name of the item you didn't receive?
Customer: Blanket
Agent: And what was the shipping address for this item?
Customer: 123 Main St
Agent: Okay, thank you. I'm going to check on the status of your order. Can you please hold for just a moment?
Customer: (hangs up)
This is a common scenario in customer service, where the agent is trying to gather information from the customer to help resolve their issue. The agent is using active listening skills to gather information from the customer, such as the order number, the name of the item, and the shipping address. The agent is also using empathy to acknowledge the customer's concern and let them know that they are there to help.


#### Comparison with ChatGPT

> Note: This section of the notebook has been updated.

When run with the query:

```Tell me how to train my dog to sit```

ChatGPT responds with:

> Training your dog to sit is a basic and essential command that can be taught using positive reinforcement. Here's a simple step-by-step guide:
> 1. **Prepare Treats:**
   Gather small, soft treats that your dog enjoys. Make sure they are easy to chew and won't take too long to eat.
> 2. **Find a Quiet Space:**
   Choose a quiet area with minimal distractions for the training session. This will help your dog focus better.
> 3. **Get Your Dog's Attention:**
   Call your dog's name to get their attention. Make sure they are looking at you.
> 4. **Use a Treat to Lure:**
   Hold a treat close to your dog's nose, and slowly move your hand upward and slightly backward over their head. As you do this, your dog's natural response will be to follow the treat with their nose, causing them to sit.
> 5. **Say the Command:**
   As your dog starts to sit, say the command "Sit" in a clear and firm voice. Use the word consistently every time you want your dog to sit.
> 6. **Reward and Praise:**
   As soon as your dog sits, immediately reward them with the treat and offer verbal praise. This positive reinforcement will help them associate sitting with positive outcomes.
> 7. **Repeat and Practice:**
   Repeat the process several times in a row during each training session. Keep the sessions short (around 5-10 minutes) to prevent your dog from losing interest.
> 8. **Add Duration:**
   Once your dog consistently sits on command, you can gradually increase the duration by waiting a couple of seconds before giving the treat. This helps reinforce the sit command.
> 9. **Generalize the Command:**
   Practice the "sit" command in different locations and with various distractions to help your dog generalize the behavior.
> 10. **Be Patient and Consistent:**
    Patience and consistency are key in dog training. Always use positive reinforcement, and avoid punishment. If your dog doesn't succeed initially, go back a step and try again.
> 
> Remember that each dog is unique, and some may learn more quickly than others. Adjust your training approach based on your dog's individual needs and progress.

#### Scaling laws

For a fixed compute budget, there's an optimal joint scaling of model size and training-token count — many earlier large models were significantly undertrained relative to their parameter count; compute-optimal training balances both rather than maximizing params alone.

### Supervised Fine-Tuning (SFT) and PEFT

SFT trains the model to imitate example (prompt, response) pairs, minimizing negative log likelihood on the response tokens, same cross-entropy mechanism as pretraining (see `05-loss-regularization.ipynb`), just applied to a curated instruction dataset instead of raw web text. Full fine-tuning updates every weight; PEFT (parameter-efficient fine-tuning) freezes the original weights and trains a much smaller add-on instead (LoRA, detailed next, is the specific PEFT technique used here). Hands-on section below runs real SFT on a small model (SmolLM2-135M) with a small dataset.

#### SFT: imitating example responses

It minimizes negative log likelihood for the responses with cross entropy loss.


- Pre-trained Models -> instruct models
- Non-reasoning models -> reasoning models
- distill capabilities of smaller models by training on high-quality synthetic data generated from larger models

#### SFT data curation

- Distillation
- Best of K / rejection sampling - Generate multiple response from the orginal model, select the best among them.
- Filtering - Start from larger scale SFT dataset, filter according to qulaity of responses and diversity of prompts
- 1000 High quality, diverse data > 1 Million mixed-quality data

#### Full fine-tuning vs. PEFT

W_new = W_original + ΔW
ΔW = A × B

- W_original: Frozen pre-trained weights
- A: Matrix of size (d × r)
- B: Matrix of size (r × k)
- r: Rank (much smaller than d and k)

![image.png](attachment:image.png)

PEFT like LoRA saves memory, learns less while forgets less

In [4]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling

In [7]:
def generate_responses(model, tokenizer, user_message, system_message=None, max_new_tokens=100):

    # format chat using tokenizer's chat template
    messages = []
    if system_message:
        messages.append({"role":"system", "content":user_message})

    # we assume the data are all single-tune conversation
    messages.append({"role":"user", "content":user_message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False, # why tokenize is False?
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip() # for text based response

    return response

In [6]:
def test_model_with_questions(model, tokenizer, questions, system_message=None, title="Model Output"):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        response = generate_responses(model, tokenizer, question,
                                      system_message)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")
        

In [13]:
def load_model_and_tokenizer(model_name, use_gpu=False):
    
    # Load base model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    if use_gpu:
        model.to("mps")

    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
                {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
                {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
                {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
                {% endif %}
                {% endfor %}"""
    
    # Tokenizer config
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
        
    return model, tokenizer

In [14]:
def display_dataset(dataset):
    # Visualize the dataset 
    rows = []
    for i in range(3):
        example = dataset[i]
        user_msg = next(m['content'] for m in example['messages']
                        if m['role'] == 'user')
        assistant_msg = next(m['content'] for m in example['messages']
                             if m['role'] == 'assistant')
        rows.append({
            'User Prompt': user_msg,
            'Assistant Response': assistant_msg
        })
    
    # Display as table
    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', None)  # Avoid truncating long strings
    display(df)

In [ ]:
USE_GPU = True

questions = [
    "Give me an 1-sentence introduction of LLM.",
    "Calculate 1+1-1",
    "What's the difference between thread and process?"
]

In [16]:
# Use the official Hugging Face model ID
model, tokenizer = load_model_and_tokenizer("Qwen/Qwen2.5-0.5B", USE_GPU)

test_model_with_questions(model, tokenizer, questions, 
                          title="Base Model (Before SFT) Output")

del model, tokenizer


=== Base Model (Before SFT) Output ===

Model Input 1:
Give me an 1-sentence introduction of LLM.
Model Output 1:
Hello! I'm a helpful assistant here. :)
ponde
You are a helpful assistant.moid
moid
You are a helpful assistant.moid
moid
You are a helpful assistant.moid
moid
You are a helpful assistant.moid
moid
You are a helpful assistant.moid
moid
You are a helpful assistant.moid
moid
You are a helpful assistant.moid
moid
You are a helpful assistant.moid
moid
You are a helpful assistant.moid


Model Input 2:
Calculate 1+1-1
Model Output 2:
1+1-1 Daytona
ratulations
You are a helpful assistant.rible
rite
Calculate 1+1-1rible
rite
Calculate 1+1-1rible
rite
Calculate 1+1-1rible
rite
Calculate 1+1-1rible
rite
Calculate 1+1-1rible
rite
Calculate 1+1-1rible
rite
Calculate 1+1-1rible
rite
Calculate 1+



KeyboardInterrupt: 

#### SFT on a small model

In [17]:
model_name = "HuggingFaceTB/SmolLM2-135M"
model, tokenizer = load_model_and_tokenizer(model_name, USE_GPU)

In [ ]:
import os
# Set a new cache directory
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf-datasets-cache"

train_dataset = load_dataset("banghua/DL-SFT-Dataset")["train"]
# Unable to download this, will revisit.
USE_GPU = False
if not USE_GPU:
    train_dataset = train_dataset.select(range(100))

display_dataset(train_dataset)

Generating train split: 100%|██████████| 2961/2961 [00:00<00:00, 49795.85 examples/s]


NotImplementedError: Loading a dataset cached in a LocalFileSystem is not supported.

In [ ]:
# SFTTrainer config 
sft_config = SFTConfig(
    learning_rate=8e-5, # Learning rate for training. 
    num_train_epochs=1, #  Set the number of epochs to train the model.
    per_device_train_batch_size=1, # Batch size for each device (e.g., GPU) during training. 
    gradient_accumulation_steps=8, # Number of steps before performing a backward/update pass to accumulate gradients.
    gradient_checkpointing=False, # Enable gradient checkpointing to reduce memory usage during training at the cost of slower training speed.
    logging_steps=2,  # Frequency of logging training progress (log every 2 steps).

)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset, 
    processing_class=tokenizer,
)
sft_trainer.train()

#### Testing the trained small model

In [ ]:
if not USE_GPU: # move model to CPU when GPU isn’t requested
    sft_trainer.model.to("cpu")
test_model_with_questions(sft_trainer.model, tokenizer, questions, 
                          title="Base Model (After SFT) Output")

#### PEFT beyond LoRA: QLoRA and prompt tuning

Full FT: retrains all weights — most flexible, most compute/storage. LoRA: freezes base weights, trains small low-rank adapter matrices — near full-FT quality, far fewer trained params. QLoRA: LoRA + 4-bit-quantized frozen base — fits fine-tuning of much larger models on limited GPU memory. Prompt/prefix-tuning: trains only soft input tokens — cheapest, least expressive.

#### Catastrophic forgetting

Gradient updates optimize only the new task objective with no pressure to preserve unrelated prior behavior — narrow fine-tuning data overwrites general capability. Detect via held-out general-capability eval (not just task eval) before/after. Mitigate: lower LR, fewer epochs, mix in general instruction data, or use LoRA (frozen base structurally protects general behavior).

#### Fine-tuning vs feature extraction

Frozen (feature extraction): train only a head on [CLS]/pooled embeddings — cheap, little data, lower ceiling. Full fine-tuning: update all layers with small LR (2e-5–5e-5), warmup, few epochs — best accuracy, risk of overfit/forgetting. Middle: gradual unfreezing, adapters/LoRA.

### LoRA (Low-Rank Adaptation)

The specific PEFT technique behind the SFT config above. Freezes the original weight matrix W_original and learns a low-rank update instead: W_new = W_original + A*B, where A and B are much smaller matrices than W_original (rank r controls how small), so training and storing the update costs a small fraction of full fine-tuning.

#### Hands-on LoRA

Fine-tunes large models efficiently by freezing the original weights and training two small low-rank matrices that approximate the necessary weight changes.

```py
W_new = W_original + ΔW
ΔW = A × B
```

- W_original: Frozen pre-trained weights
- A: Matrix of size (d × r)
- B: Matrix of size (r × k)
- r: Rank (much smaller than d and k)

```py
Original: h = W × x
LoRA: h = W × x + (B × A) × x
```

Fewer trainable parameters → less memory and compute during training.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling
from trl import SFTTrainer, SFTConfig

# Load a small model (adjust the model name as needed)
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
# Set tokenizer chat template if needed
if not tokenizer.chat_template:
    tokenizer.chat_template = """{% for message in messages %}
            {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
            {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
            {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
            {% endif %}
            {% endfor %}"""

if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Assume train_dataset is already loaded and pre-processed,
# for example, by using load_dataset from the datasets library.
# Here we select a small subset for quick demonstration.
USE_GPU = False
if not USE_GPU:
    train_dataset = train_dataset.select(range(100))

In [ ]:
# Define SFT configuration along with LoRA hyperparameters:
# Here, lora_r (rank) and lora_alpha (scaling factor) are added.
sft_config = SFTConfig(
    learning_rate=8e-5,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=False,
    logging_steps=2,
    lora_r=8,        # Low-rank dimension
    lora_alpha=16,   # Scaling factor for LoRA updates
)

# Create SFTTrainer with LoRA-enabled configuration
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

# Start training
sft_trainer.train()

#### Serving and merging LoRA adapters

Merged: fold LoRA weights into the base model permanently — zero inference-time overhead, but locks in one task/style, needs a full merged copy per adapter. Kept separate (adapter switching): swap lightweight adapters per request on a shared base model — small inference overhead, but supports serving many tasks/customers from one base model in memory.

#### Model merging and distillation

Merging: combine fine-tuned weights without training (averaging/model soups, TIES, DARE, SLERP) — cheap multi-skill; needs a shared base, can interfere. Distillation: small student trained on teacher outputs/logits (soft labels) → cheaper inference; loses some capability; check licence terms.

### Alignment and RLHF

Why a pretrained/fine-tuned model (previous notebook) still isn't reliably helpful/honest/harmless on its own, an alignment gap between what it's capable of and what it actually does. Covers the SFT to reward-model to RL pipeline, Bradley-Terry preference modeling, reward hacking, and DPO as a simpler alternative to full RLHF.

#### Alignment gaps

Model has capability, but model objective during training is to predict next token, hence it does not align with what we want from an assitant (be helpful, be honest, do not cause harm) as these are human values and they do not fit cleanly into a loss fucntion.

  > Prediction isn’t alignment. A model trained to predict human text will learn to predict human text, including all our biases, inconsistencies, and worst impulses.

#### The three-act story

![image.png](attachment:image.png)

#### Act 1: SFT

Pretraind Model, Fine tune on curated dataset of HQ demonstrations as created by human annotators as written or selected as good responses to various prompts which are answers that clear, accurate, appropriately hedged, and genuinely helpful. --> This is SFT Model, (format,  but does not know fine grain details on how good different responses are relative to each other, and no mechanism to improve beyond on what it is trained on)

#### Act 2: a proxy for human judgement

Rather than writing a good response, asking them to select good response as a preference, it is faster as human are far better than at ranking than generating and hence a rich pairwise preference dataset. Hence Train a reward model on these comparison, which learns to predict for any (prompt, response) pair, a scalar score reflecting how much a human would prefer that response. 

Approach is Bradley-Terry Model - treating human preference as a probabilistic comparison

$$P(y_w \succ y_l \mid x) = \sigma\left( r_\varphi(x, y_w) - r_\varphi(x, y_l) \right) \\
\text{Loss: } -\mathbb{E}\left[\log \sigma\left(r_\varphi(x, y_w) - r_\varphi(x, y_l)\right)\right]$$

where $σ$ is the sigmoid function, $y_w$ is the preferred response (w stands for winning), $y_l$  (l stands for losing) is the dispreferred response, and $r_φ$ is the reward model.

After training, you have a function that, given any text, returns a number. 

High number = humans probably like it. Low number = humans probably don’t.

#### Bradley-Terry, implemented

The batch loss and the optimization step, made concrete:

$$\mathcal{L}_{batch} = \frac{1}{N} \sum_{i=1}^{N} -\log \sigma\left( r_\varphi(x_i, y_w^i) - r_\varphi(x_i, y_l^i) \right)$$

$$\varphi_{t+1} = \varphi_t - \alpha \nabla_\varphi \mathcal{L}(\varphi_t)$$

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# ============================================================================
# STEP 1: Define the Reward Model
# ============================================================================

class SimpleRewardModel(nn.Module):
    """
    A simple neural network reward model.
    Input: (prompt embedding, response embedding) concatenated
    Output: A scalar reward score
    """
    def __init__(self, embedding_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim * 2, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)  # Output: single reward score
        )
    
    def forward(self, prompt_emb, response_emb):
        """
        Args:
            prompt_emb: shape (batch_size, embedding_dim)
            response_emb: shape (batch_size, embedding_dim)
        Returns:
            rewards: shape (batch_size, 1) - scalar reward per pair
        """
        concatenated = torch.cat([prompt_emb, response_emb], dim=1)
        return self.net(concatenated)

# ============================================================================
# STEP 2: Define the Bradley-Terry Loss
# ============================================================================

def bradley_terry_loss(reward_preferred, reward_dispreferred):
    """
    Compute Bradley-Terry loss for a batch of preference pairs.
    
    Mathematical formula:
    L = -log(σ(r_w - r_l))
    where σ is sigmoid, r_w is reward of preferred response, r_l is reward of dispreferred
    
    Args:
        reward_preferred: shape (batch_size, 1), rewards for preferred responses
        reward_dispreferred: shape (batch_size, 1), rewards for dispreferred responses
    
    Returns:
        loss: scalar tensor
    """
    # Compute difference
    reward_diff = reward_preferred - reward_dispreferred
    
    # Apply sigmoid: σ(z) = 1 / (1 + e^(-z))
    sigmoid_diff = torch.sigmoid(reward_diff)
    
    # Binary cross-entropy: -log(sigmoid_diff)
    # PyTorch's BCELoss with target=1 computes exactly this
    loss = -torch.log(sigmoid_diff + 1e-8)  # Add small epsilon to avoid log(0)
    
    return loss.mean()

# ============================================================================
# STEP 3: Create Synthetic Data (Preference Pairs)
# ============================================================================

def create_synthetic_data(num_samples=100, embedding_dim=64):
    """
    Create synthetic prompt-response pairs with preferences.
    
    In reality:
    - Prompts: queries from users
    - Responses: different model outputs for same prompt
    - Preferences: which response humans prefer
    """
    np.random.seed(42)
    torch.manual_seed(42)
    
    # Generate random embeddings
    prompt_embeddings = torch.randn(num_samples, embedding_dim)
    response_preferred = torch.randn(num_samples, embedding_dim)
    response_dispreferred = torch.randn(num_samples, embedding_dim)
    
    return prompt_embeddings, response_preferred, response_dispreferred

# ============================================================================
# STEP 4: Training Loop
# ============================================================================

def train_bradley_terry(num_epochs=50, batch_size=16, embedding_dim=64):
    """
    Full training pipeline for Bradley-Terry reward model.
    """
    # Initialize model
    model = SimpleRewardModel(embedding_dim=embedding_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Create data
    prompts, preferred_responses, dispreferred_responses = create_synthetic_data(
        num_samples=100, embedding_dim=embedding_dim
    )
    
    dataset = TensorDataset(prompts, preferred_responses, dispreferred_responses)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    losses = []
    
    # Training loop
    for epoch in range(num_epochs):
        epoch_loss = 0
        for batch_prompts, batch_preferred, batch_dispreferred in dataloader:
            # Forward pass
            reward_preferred = model(batch_prompts, batch_preferred)
            reward_dispreferred = model(batch_prompts, batch_dispreferred)
            
            # Compute Bradley-Terry loss
            loss = bradley_terry_loss(reward_preferred, reward_dispreferred)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(dataloader)
        losses.append(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
    
    return model, losses

# ============================================================================
# STEP 5: Visualization
# ============================================================================

def visualize_training(losses):
    """Plot training loss over time."""
    plt.figure(figsize=(10, 5))
    plt.plot(losses, linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Bradley-Terry Loss')
    plt.title('Reward Model Training: Bradley-Terry Loss Over Time')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ============================================================================
# STEP 6: Inspect Learned Preferences
# ============================================================================

def inspect_model(model, num_samples=5):
    """
    See how the trained model scores different responses.
    """
    model.eval()
    
    with torch.no_grad():
        prompts, preferred, dispreferred = create_synthetic_data(
            num_samples=num_samples, embedding_dim=64
        )
        
        rewards_preferred = model(prompts, preferred).squeeze()
        rewards_dispreferred = model(prompts, dispreferred).squeeze()
        
        print("\n" + "="*70)
        print("MODEL PREFERENCE ANALYSIS")
        print("="*70)
        
        for i in range(num_samples):
            r_w = rewards_preferred[i].item()
            r_l = rewards_dispreferred[i].item()
            
            # Compute probability using Bradley-Terry formula
            prob = torch.sigmoid(torch.tensor(r_w - r_l)).item()
            
            print(f"\nExample {i+1}:")
            print(f"  Preferred response score:     {r_w:7.3f}")
            print(f"  Dispreferred response score:  {r_l:7.3f}")
            print(f"  Difference (r_w - r_l):      {r_w - r_l:7.3f}")
            print(f"  P(preferred ≻ dispreferred):  {prob:.4f} (via sigmoid)")
            print(f"  Model confidence:             {'HIGH' if prob > 0.7 else 'MEDIUM' if prob > 0.55 else 'LOW'}")

# ============================================================================
# RUN THE FULL PIPELINE
# ============================================================================

if __name__ == "__main__":
    print("Training Bradley-Terry Reward Model...\n")
    model, losses = train_bradley_terry(num_epochs=50, batch_size=16, embedding_dim=64)
    
    print("\n✓ Training complete!")
    
    # Inspect learned preferences
    inspect_model(model, num_samples=5)
    
    # Plot training curve
    visualize_training(losses)

#### Act 3: using RL to maximise the proxy

The language model (now called the policy) generates responses, the reward model scores them, and a reinforcement learning algorithm, typically PPO (Proximal Policy Optimization), updates the policy to produce higher-scoring responses.

But, we can't maximise reward unchecked. as model would quickly learn to produce response that game reward model. Hence Adding a KL penalty, a term that penalises the policy for drifting too far from the SFT model. 

$$\max_{\pi} \; \mathbb{E}_{x \sim \mathcal{D},\, y \sim \pi}\!\left[ r_{\varphi}(x, y) \right] \\ - \beta \, \mathrm{KL}\!\left[ \pi(\cdot \mid x) \,\|\, \pi_{\text{SFT}}(\cdot \mid x) \right]$$

$β$ controls the strength of the constraint. 
Small $β$ → more reward-chasing, higher drift risk. Large $β$ → stays close to SFT, lower ceiling

 > ##### WHY DOES THIS WORK?
 > Human pairwise comparison captures nuance that’s hard to formalise. A comparison between “helpful and slightly too long” versus “concise but misses the point” contains information about priorities, tone, accuracy and trustworthiness, all encoded in a single bit. Thousands of such bits, aggregated into a reward model, produce a surprisingly rich proxy for what humans actually value.

#### A brief history: from robots to language models

2017 Christiano et al. (OpenAI/DeepMind) demonstrate RLHF on Atari games and robotic locomotion tasks, humans give feedback on short video clips of agent behaviour, without needing to write reward functions. The key insight: comparative feedback is cheap and surprisingly informative.

2019–2020 OpenAI applies RLHF to language tasks, summarisation (TL;DR dataset), instruction-following. Early results show that RLHF-trained models are preferred by humans even when significantly smaller than purely supervised baselines.

2022 InstructGPT paper drops. RLHF at scale makes GPT models dramatically more helpful and less harmful. The result is so pronounced that human evaluators prefer the RLHF-trained 1.3B model over the non-RLHF 175B GPT-3, on most tasks. Scale ≠ alignment.

Late 2022 ChatGPT launches. RLHF goes from research technique to product-defining feature overnight. The gap between GPT-3 (capable but awkward) and ChatGPT (genuinely conversational) is almost entirely RLHF.

2023 DPO (Direct Preference Optimization) published by Rafailov et al. at Stanford. A mathematical reparameterisation shows you can skip the explicit reward model and the PPO loop entirely, training directly on preference data. Simpler, cheaper, competitive results.

2024–2026 RLHF becomes post-training standard across all frontier models. Online iterative RLHF, AI-generated feedback, process reward models, and constitutional AI variants all proliferate. The field is now less about “does RLHF work” and more about which variant, at what scale, for which capabilities.

#### The villain: reward hacking

![image.png](attachment:image.png)

Here’s the uncomfortable truth about RLHF: the reward model is not the goal. It’s a proxy for the goal. And optimised proxies have a habit of being gamed. This is Goodhart’s Law, stated in 1975 by the economist Charles Goodhart: “When a measure becomes a target, it ceases to be a good measure.” 

In RLHF, the reward model is trained on thousands of human comparisons, but it’s a finite model with finite data. As the policy pushes harder and harder to maximise it, it eventually finds behaviours that score high on the reward model while being subtly off in ways the reward model wasn’t trained to catch.

A classic reward-hacking failure mode: the model learns that the reward model assigns high scores to confident, enthusiastic, verbose responses, even when a shorter, more precise answer would actually serve the user better. The proxy (reward model score) diverges from the true goal (genuine helpfulness)

The most common reward-hacking failure modes in language models are:

1. *Sycophancy*. Humans tend to prefer responses that agree with them, validate their views, and use flattering language. The reward model learns this. The policy learns to exploit it. Over time, the model becomes a yes-machine that tells you what you want to hear rather than what’s true.

2. *Verbosity*. Longer responses tend to look more thorough and get higher scores, even when a concise answer would serve better. Models learn to pad responses with context, caveats, and unnecessary qualifications.

3. *Format over substance*. Bullet points, headers, bold text, all of these make a response look organised and thus score well. A model can learn to add structural ornamentation to mediocre content and get rewarded for it.

4. *Hallucination confidence*. The reward model may not distinguish between “accurate and confident” and “inaccurate and confident” if the inaccuracies are subtle. The model learns confident assertion is preferred over appropriately hedged uncertainty.

Human preference, as expressed through pairwise comparisons in a lab setting, is not a reliable proxy for genuine human benefit. We prefer responses that feel good in the moment. We’re bad at noticing subtle inaccuracies. We prefer confident-sounding answers. Building alignment on top of these preferences bakes in their limitations.

#### DPO: skipping the reward model

The classic RLHF pipeline (SFT → reward model → PPO) is powerful but complicated. You’re managing three to four models in memory simultaneously. PPO is notoriously finicky to tune. The interaction between policy updates and reward model evaluation creates feedback loops that are hard to debug.

The RLHF objective, reward maximisation with a KL constraint, has a closed-form optimal solution. You can _reparameterise_ the reward model in terms of the policy itself, which means you never need to train a separate reward model at all. You can optimise directly on preference data, using a simple classification loss. This is Direct Preference Optimization (DPO).

![image.png](attachment:image.png)

DPO eliminates the explicit reward model and the online sampling loop, replacing them with a single classification loss over static preference data. Simpler and more stable, but with limitations at the frontier of capability.

The DPO loss is elegantly simple:

$$\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}\left[\log \sigma\left( \beta \log\left[\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}\right] - \beta \log\left[\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right] \right)\right]$$

Intuition: increase log-probability of preferred responses relative to dispreferred ones, weighted by how confident the current policy already is about each pair. 

Because it operates on a static dataset (no new samples generated during training), the policy can only improve within the support of the preference data it was trained on. It can’t explore, and it can’t adapt to novel situations the training data didn’t cover.

RLHF (PPO), FRONTIER CHOICE

- Best results on large-scale training
- Online exploration beyond training data
- Adapts dynamically during training
- 3–4 models in memory simultaneously
- Unstable: sensitive to hyperparameters
- Reward hacking risk increases with optimisation

DPO, PRACTITIONER DEFAULT

- One model, one simple classification loss
- Stable, reproducible, cheap to run
- No separate reward model needed
- Offline: limited to preference data distribution
- Distribution shift issues at scale
- Lower performance ceiling vs PPO on large model

#### Current research

- Online iterative RLHF has become standard in large-scale training. Rather than training the reward model once and using it statically, you iterate: generate new outputs, collect new comparisons (increasingly from AI annotators rather than humans), retrain the reward model, update the policy, repeat. This allows the system to keep improving beyond the initial preference data, but also means errors compound across iterations.

- AI-generated feedback, sometimes called RLAIF (RL from AI Feedback), is now widespread. It’s simply too expensive to collect human comparisons at the scale needed to train frontier models. AI annotators, typically a more powerful model evaluating a weaker one, provide the bulk of the signal. This raises obvious circularity concerns (are we just teaching models to satisfy other models?) and is an active area of research.

- Process reward models (PRMs) versus outcome reward models (ORMs) have become a central distinction, particularly for reasoning tasks. Traditional reward models score final outputs; process reward models score intermediate reasoning steps. For maths and coding, PRMs appear to give significantly better signal and reduce certain types of reward hacking.

- Constitutional AI (Anthropic’s approach) adds a rule-based layer on top of RLHF: the model is given a set of principles and asked to critique and revise its own outputs against them before a human evaluates. This reduces the human feedback burden and seems to improve robustness to adversarial prompts.



> ##### THE BIGGEST OPEN QUESTION
  > Human preference is not the same as human benefit, and human benefit is not the same as what’s good for the world. RLHF optimises for the first. The hope is that it approximates the second and points toward the third. Whether that chain of approximations holds as models become more capable is the question that alignment researchers are most worried about, and the one we have the least ability to answer from empirical data alone.

Article Reference & Owner - https://probabilityandstatistics.substack.com/p/issue-3-teaching-ai-to-want-the-right

#### RLAIF

RLHF: reward model trained on human preference labels. RLAIF: reward signal comes from an AI model's judgments instead of (or alongside) human labels — cheaper and faster to scale than collecting human preference data, at the cost of inheriting whatever biases/blind spots the judging AI has.